# Notebook 03 — Train All Models (3 Models × 3 Aug Modes = 9 Experiments)

**Run after Notebooks 01 and 02.**

Experiments:
| Exp | Model | Augmentation |
|-----|-------|-------------|
| 01  | EfficientNet-B0 | No Aug |
| 02  | EfficientNet-B0 | GIN Only |
| 03  | EfficientNet-B0 | Full Aug |
| 04  | ResNet50 | No Aug |
| 05  | ResNet50 | GIN Only |
| 06  | ResNet50 | Full Aug |
| 07  | MedSAM ViT-B | No Aug |
| 08  | MedSAM ViT-B | GIN Only |
| 09  | MedSAM ViT-B | Full Aug |

Each cell runs one experiment. Run sequentially or skip any you don't need.
Results are saved to `experiments/<exp_name>/`.

In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Setup
# ─────────────────────────────────────────────
import sys, json, yaml
from pathlib import Path
import pandas as pd
import torch

PROJECT_ROOT = Path('.').resolve().parent
paths = json.loads((PROJECT_ROOT / 'paths.json').read_text())

FED7_ROOT    = Path(paths['fed7_root'])
PREPROCESSED = Path(paths['preprocessed'])
SPLITS_DIR   = PROJECT_ROOT / 'splits'
EXPERIMENTS  = Path(paths['experiments'])
MEDSAM_CKPT  = paths['medsam_ckpt']
CONFIGS_DIR  = PROJECT_ROOT / 'configs'

# Add both project roots to path
for p in [str(PROJECT_ROOT), str(FED7_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Load splits
train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df   = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df  = pd.read_csv(SPLITS_DIR / 'test.csv')

print(f'Train  : {len(train_df)} slices | {train_df["patient_id"].nunique()} patients')
print(f'Val    : {len(val_df)} slices   | {val_df["patient_id"].nunique()} patients')
print(f'Test   : {len(test_df)} slices  | {test_df["patient_id"].nunique()} patients')

Device : cuda
GPU    : NVIDIA GeForce RTX 4070
VRAM   : 12.9 GB
Train  : 2287 slices | 157 patients
Val    : 268 slices   | 22 patients
Test   : 521 slices  | 42 patients


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Helper: build dataloaders
# ─────────────────────────────────────────────
from datasets.fedbca_dataset import FedBCaDataset, build_dataloader

def make_loaders(model_type, batch_size, num_workers=4,
                 use_aug=True, gin_only=False, resnet_aug=False):
    train_ds = FedBCaDataset(
        train_df, str(PREPROCESSED), mode='train',
        model_type=model_type,
        use_augmentation=use_aug,
        use_gin_only=gin_only,
        use_resnet_aug=resnet_aug,
    )
    val_ds = FedBCaDataset(
        val_df, str(PREPROCESSED), mode='val',
        model_type=model_type, use_augmentation=False,
    )
    tl = build_dataloader(train_ds, batch_size, num_workers, use_sampler=True)
    vl = build_dataloader(val_ds,   batch_size, num_workers, use_sampler=False)
    print(f'  Train batches: {len(tl)} | Val batches: {len(vl)}')
    return tl, vl

print('Dataloader helper ready.')

Dataloader helper ready.


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Helper: build criterion from config
# ─────────────────────────────────────────────
from utils.losses import JointLoss

def make_criterion(config):
    return JointLoss(
        seg_weight     = config.get('seg_weight', 0.85),
        cls_weight     = config.get('cls_weight', 0.15),
        focal_gamma    = config.get('focal_gamma', 2.0),
        mibc_alpha     = config.get('mibc_alpha', 1.4),
        ds_weights     = tuple(config.get('ds_weights', [0.50, 0.25])),
        tversky_weight = config.get('tversky_weight', 0.20),
    )

print('Criterion helper ready.')

Criterion helper ready.


In [4]:
# ─────────────────────────────────────────────
# EXP 01 — EfficientNet, No Augmentation
# Expected: DSC 0.55-0.65, AUC 0.76-0.83
# Time: ~25 min (RTX 4070)
# ─────────────────────────────────────────────
from models.efficientnet.model import EfficientNetMultiTask
from utils.trainer import train_model

with open(CONFIGS_DIR / 'efficientnet.yaml') as f:
    cfg_eff = yaml.safe_load(f)

tl, vl = make_loaders('cnn', cfg_eff['batch_size'], use_aug=False)
model  = EfficientNetMultiTask(pretrained=True)
crit   = make_criterion(cfg_eff)

r01 = train_model(model, tl, vl, crit, cfg_eff,
                  'exp01_eff_noaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP01: DSC={r01["best_dsc"]:.4f} | AUC={r01["best_auc"]:.4f}')

  Train batches: 285 | Val batches: 34


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)


[EfficientNet-B0-CORRECTED] in_ch=5 | params=4,169,692

[exp01_eff_noaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 180
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/180 | Loss=1.1699 (seg=1.376 cls=0.000) | tDSC=0.3550 AUC=0.6161 comb=0.4725★ | LR=1.50e-05 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 175/180 | Loss=0.5796 (seg=0.678 cls=0.020) | tDSC=0.7274 AUC=0.8125 comb=0.7657 | LR=1.56e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 180/180 | Loss=0.5958 (seg=0.698 cls=0.015) | tDSC=0.7284 AUC=0.8036 comb=0.7622 | LR=1.50e-06 | cls_w=0.150

[exp01_eff_noaug] Done.
  Best DSC : 0.7333
  Best AUC : 0.8125
  Combined : 0.7690

EXP01: DSC=0.7333 | AUC=0.8125


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes

In [5]:
# ─────────────────────────────────────────────
# EXP 02 — EfficientNet, GIN Only
# Expected: DSC 0.57-0.67, AUC 0.78-0.85
# Time: ~28 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_eff['batch_size'], use_aug=True, gin_only=True)
model  = EfficientNetMultiTask(pretrained=True)
crit   = make_criterion(cfg_eff)

r02 = train_model(model, tl, vl, crit, cfg_eff,
                  'exp02_eff_ginonly', str(EXPERIMENTS), DEVICE)
print(f'\nEXP02: DSC={r02["best_dsc"]:.4f} | AUC={r02["best_auc"]:.4f}')
print(f'GIN gain (EfficientNet): ΔDSC={r02["best_dsc"]-r01["best_dsc"]:+.4f} | '
      f'ΔAUC={r02["best_auc"]-r01["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)


[EfficientNet-B0-CORRECTED] in_ch=5 | params=4,169,692

[exp02_eff_ginonly] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 180
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/180 | Loss=1.3156 (seg=1.548 cls=0.000) | tDSC=0.1603 AUC=0.3571 comb=0.2489★ | LR=1.50e-05 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 175/180 | Loss=0.8770 (seg=1.028 cls=0.019) | tDSC=0.7629 AUC=0.8571 comb=0.8053 | LR=1.56e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 180/180 | Loss=0.8863 (seg=1.038 cls=0.026) | tDSC=0.7634 AUC=0.8571 comb=0.8056★ | LR=1.50e-06 | cls_w=0.150

[exp02_eff_ginonly] Done.
  Best DSC : 0.7634
  Best AUC : 0.8571
  Combined : 0.8056

EXP02: DSC=0.7634 | AUC=0.8571
GIN gain (EfficientNet): ΔDSC=+0.0301 | ΔAUC=+0.0446


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes

In [6]:
# ─────────────────────────────────────────────
# EXP 03 — EfficientNet, Full Augmentation
# Expected: DSC 0.60-0.70, AUC 0.80-0.87
# Time: ~30 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_eff['batch_size'], use_aug=True)
model  = EfficientNetMultiTask(pretrained=True)
crit   = make_criterion(cfg_eff)

r03 = train_model(model, tl, vl, crit, cfg_eff,
                  'exp03_eff_fullaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP03: DSC={r03["best_dsc"]:.4f} | AUC={r03["best_auc"]:.4f}')
print(f'Full aug gain (vs no aug): '
      f'ΔDSC={r03["best_dsc"]-r01["best_dsc"]:+.4f} | '
      f'ΔAUC={r03["best_auc"]-r01["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)


[EfficientNet-B0-CORRECTED] in_ch=5 | params=4,169,692

[exp03_eff_fullaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 180
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/180 | Loss=1.2884 (seg=1.516 cls=0.000) | tDSC=0.2360 AUC=0.4911 comb=0.3508★ | LR=1.50e-05 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 175/180 | Loss=0.8287 (seg=0.967 cls=0.045) | tDSC=0.7904 AUC=0.8482 comb=0.8164 | LR=1.56e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 180/180 | Loss=0.8422 (seg=0.981 cls=0.056) | tDSC=0.7932 AUC=0.8304 comb=0.8099 | LR=1.50e-06 | cls_w=0.150

[exp03_eff_fullaug] Done.
  Best DSC : 0.7890
  Best AUC : 0.8571
  Combined : 0.8197

EXP03: DSC=0.7890 | AUC=0.8571
Full aug gain (vs no aug): ΔDSC=+0.0557 | ΔAUC=+0.0446


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes

In [9]:
# ─────────────────────────────────────────────
# EXP 04 — ResNet50, No Augmentation
# Expected: DSC 0.60-0.70, AUC 0.78-0.85
# Time: ~35 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()
from models.resnet50.model import ResNet50MultiTask

with open(CONFIGS_DIR / 'resnet50.yaml') as f:
    cfg_res = yaml.safe_load(f)

tl, vl = make_loaders('cnn', cfg_res['batch_size'], use_aug=False)
model  = ResNet50MultiTask(pretrained=True, use_mixstyle=cfg_res.get('use_mixstyle', True))
crit   = make_criterion(cfg_res)

r04 = train_model(model, tl, vl, crit, cfg_res,
                  'exp04_res50_noaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP04: DSC={r04["best_dsc"]:.4f} | AUC={r04["best_auc"]:.4f}')

  Train batches: 285 | Val batches: 34
[ResNet50-V5] in_ch=5 | mixstyle=True | params=45,884,872

[exp04_res50_noaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 140
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/140 | Loss=0.8860 (seg=1.042 cls=0.000) | tDSC=0.6707 AUC=0.5536 comb=0.6180★ | LR=2.51e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/140 | Loss=0.6782 (seg=0.798 cls=0.000) | tDSC=0.7402 AUC=0.5446 comb=0.6522★ | LR=5.01e-06 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  90/140 | Loss=0.4333 (seg=0.505 cls=0.029) | tDSC=0.7385 AUC=0.8125 comb=0.7718 | LR=4.02e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  95/140 | Loss=0.4259 (seg=0.496 cls=0.029) | tDSC=0.7363 AUC=0.8214 comb=0.7746 | LR=3.43e-06 | cls_w=0.150

[EARLY STOP] No improvement for 30 epochs.

[exp04_res50_noaug] Done.
  Best DSC : 0.7393
  Best AUC : 0.8304
  Combined : 0.7803

EXP04: DSC=0.7393 | AUC=0.8304


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes

In [6]:
with open(CONFIGS_DIR / 'resnet50.yaml') as f:
    cfg_res = yaml.safe_load(f)

In [7]:
# ─────────────────────────────────────────────
# EXP 05 — ResNet50, GIN Only
# Expected: DSC 0.62-0.72, AUC 0.80-0.87
# Time: ~38 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()
from utils.trainer import train_model
from models.resnet50.model import ResNet50MultiTask
# ResNet50 uses ResNet50AugmentationPipeline which includes GIN
# use_resnet_aug=True activates ResNet50-specific GIN pipeline
tl, vl = make_loaders('cnn', cfg_res['batch_size'],
                       use_aug=True, gin_only=True)
model  = ResNet50MultiTask(pretrained=True, use_mixstyle=cfg_res.get('use_mixstyle', True))
crit   = make_criterion(cfg_res)

r05 = train_model(model, tl, vl, crit, cfg_res,
                  'exp05_res50_ginonly', str(EXPERIMENTS), DEVICE)
print(f'\nEXP05: DSC={r05["best_dsc"]:.4f} | AUC={r05["best_auc"]:.4f}')
print(f'GIN gain (ResNet50): ΔDSC={r05["best_dsc"]-r04["best_dsc"]:+.4f} | '
      f'ΔAUC={r05["best_auc"]-r04["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34
[ResNet50-V5] in_ch=5 | mixstyle=True | params=45,884,872


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)



[exp05_res50_ginonly] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 140
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/140 | Loss=0.9103 (seg=1.071 cls=0.000) | tDSC=0.6612 AUC=0.4375 comb=0.5605★ | LR=2.51e-06 | cls_w=0.000


...

Ep  75/140 | Loss=0.4265 (seg=0.496 cls=0.034) | tDSC=0.7181 AUC=0.8571 comb=0.7807 | LR=5.87e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  80/140 | Loss=0.4264 (seg=0.497 cls=0.025) | tDSC=0.7161 AUC=0.8571 comb=0.7796 | LR=5.25e-06 | cls_w=0.150

[EARLY STOP] No improvement for 30 epochs.

[exp05_res50_ginonly] Done.
  Best DSC : 0.7258
  Best AUC : 0.8661
  Combined : 0.7889

EXP05: DSC=0.7258 | AUC=0.8661


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes

NameError: name 'r04' is not defined

In [8]:
# ─────────────────────────────────────────────
# EXP 06 — ResNet50, Full Augmentation
# Expected: DSC 0.65-0.76, AUC 0.82-0.88
# Time: ~40 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_res['batch_size'],
                       use_aug=True, resnet_aug=True)
model  = ResNet50MultiTask(pretrained=True, use_mixstyle=cfg_res.get('use_mixstyle', True))
crit   = make_criterion(cfg_res)

r06 = train_model(model, tl, vl, crit, cfg_res,
                  'exp06_res50_fullaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP06: DSC={r06["best_dsc"]:.4f} | AUC={r06["best_auc"]:.4f}')
print(f'Full aug gain (ResNet50): '
      f'ΔDSC={r06["best_dsc"]-r04["best_dsc"]:+.4f} | '
      f'ΔAUC={r06["best_auc"]-r04["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34
[ResNet50-V5] in_ch=5 | mixstyle=True | params=45,884,872

[exp06_res50_fullaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 140
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/140 | Loss=0.8864 (seg=1.043 cls=0.000) | tDSC=0.6302 AUC=0.4196 comb=0.5355★ | LR=2.51e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/140 | Loss=0.7461 (seg=0.878 cls=0.000) | tDSC=0.7216 AUC=0.4732 comb=0.6098★ | LR=5.01e-06 | cls_w=0.000


...

Ep 110/140 | Loss=0.5151 (seg=0.592 cls=0.081) | tDSC=0.7726 AUC=0.8036 comb=0.7865 | LR=1.89e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 115/140 | Loss=0.4898 (seg=0.561 cls=0.086) | tDSC=0.7717 AUC=0.7946 comb=0.7820 | LR=1.48e-06 | cls_w=0.150

[EARLY STOP] No improvement for 30 epochs.

[exp06_res50_fullaug] Done.
  Best DSC : 0.7766
  Best AUC : 0.8125
  Combined : 0.7928

EXP06: DSC=0.7766 | AUC=0.8125


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes

NameError: name 'r04' is not defined

In [10]:
# ─────────────────────────────────────────────
# EXP 07 — MedSAM ViT-B, No Augmentation
# Expected: DSC 0.68-0.78, AUC 0.82-0.88
# Time: ~60 min (ViT-B is larger)
# VRAM: ~7.5 GB with AMP
# ─────────────────────────────────────────────
torch.cuda.empty_cache()
from models.medsam.model import MedSAMMultiTask
from utils.trainer import train_model

with open(CONFIGS_DIR / 'medsam.yaml') as f:
    cfg_msam = yaml.safe_load(f)

# Override checkpoint path from paths.json
cfg_msam['medsam_checkpoint'] = MEDSAM_CKPT

assert Path(MEDSAM_CKPT).exists(), (
    f'MedSAM checkpoint not found: {MEDSAM_CKPT}\n'
    'Download from: https://github.com/bowang-lab/MedSAM'
)

tl, vl = make_loaders('cnn', cfg_msam['batch_size'], use_aug=False)
model  = MedSAMMultiTask(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
    freeze_encoder=False,
)
crit   = make_criterion(cfg_msam)

r07 = train_model(model, tl, vl, crit, cfg_msam,
                  'exp07_medsam_noaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP07: DSC={r07["best_dsc"]:.4f} | AUC={r07["best_auc"]:.4f}')

  Train batches: 571 | Val batches: 67


c:\Users\Mayank\miniconda3\envs\medsam_fedbca\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


[MedSAM] resized pos_embed -> torch.Size([1, 16, 16, 768])
[MedSAM-MultiTask] in_ch=5 | freeze_enc=False | params=88,483,004

[exp07_medsam_noaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 200
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/200 | Loss=1.4633 (seg=1.721 cls=0.000) | tDSC=0.1575 AUC=0.6696 comb=0.3880★ | LR=2.70e-06 | cls_w=0.000


...

Ep 125/200 | Loss=0.8228 (seg=0.946 cls=0.123) | tDSC=0.6622 AUC=0.8125 comb=0.7298 | LR=3.02e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 130/200 | Loss=0.8342 (seg=0.961 cls=0.114) | tDSC=0.6668 AUC=0.8125 comb=0.7323 | LR=2.72e-06 | cls_w=0.150

[EARLY STOP] No improvement for 80 epochs.


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes


[exp07_medsam_noaug] Done.
  Best DSC : 0.6643
  Best AUC : 0.8214
  Combined : 0.7350

EXP07: DSC=0.6643 | AUC=0.8214


In [11]:
from models.medsam.model import MedSAMMultiTask

m = MedSAMMultiTask(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
    freeze_encoder=False,
)

print("Image size:", m.image_size if hasattr(m, "image_size") else "NO_ATTR")

print("Pos embed shape:")
print(m.image_encoder.pos_embed.shape)

c:\Users\Mayank\miniconda3\envs\medsam_fedbca\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


[MedSAM] resized pos_embed -> torch.Size([1, 16, 16, 768])
[MedSAM-MultiTask] in_ch=5 | freeze_enc=False | params=88,483,004
Image size: NO_ATTR
Pos embed shape:
torch.Size([1, 16, 16, 768])


In [12]:
# ─────────────────────────────────────────────
# EXP 08 — MedSAM ViT-B, GIN Only
# Expected: DSC 0.70-0.80, AUC 0.84-0.89
# Key experiment: does GIN help medically pretrained model?
# Time: ~65 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_msam['batch_size'],
                       use_aug=True, gin_only=True)
model  = MedSAMMultiTask(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
    freeze_encoder=False,
)
crit   = make_criterion(cfg_msam)

r08 = train_model(model, tl, vl, crit, cfg_msam,
                  'exp08_medsam_ginonly', str(EXPERIMENTS), DEVICE)
print(f'\nEXP08: DSC={r08["best_dsc"]:.4f} | AUC={r08["best_auc"]:.4f}')
print(f'GIN gain (MedSAM): ΔDSC={r08["best_dsc"]-r07["best_dsc"]:+.4f} | '
      f'ΔAUC={r08["best_auc"]-r07["best_auc"]:+.4f}')

  Train batches: 571 | Val batches: 67


c:\Users\Mayank\miniconda3\envs\medsam_fedbca\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


[MedSAM] resized pos_embed -> torch.Size([1, 16, 16, 768])
[MedSAM-MultiTask] in_ch=5 | freeze_enc=False | params=88,483,004

[exp08_medsam_ginonly] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 200
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/200 | Loss=1.4104 (seg=1.659 cls=0.000) | tDSC=0.2130 AUC=0.5714 comb=0.3743★ | LR=2.70e-06 | cls_w=0.000


...

Ep 140/200 | Loss=0.7188 (seg=0.824 cls=0.122) | tDSC=0.6456 AUC=0.7768 comb=0.7046 | LR=2.14e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 145/200 | Loss=0.7160 (seg=0.821 cls=0.121) | tDSC=0.6451 AUC=0.7768 comb=0.7043 | LR=1.87e-06 | cls_w=0.150

[EARLY STOP] No improvement for 80 epochs.


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes


[exp08_medsam_ginonly] Done.
  Best DSC : 0.6510
  Best AUC : 0.8214
  Combined : 0.7277

EXP08: DSC=0.6510 | AUC=0.8214
GIN gain (MedSAM): ΔDSC=-0.0133 | ΔAUC=+0.0000


In [13]:
# ─────────────────────────────────────────────
# EXP 09 — MedSAM ViT-B, Full Augmentation (MAIN MODEL)
# Expected: DSC 0.75-0.85, AUC 0.85-0.91
# Time: ~70 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_msam['batch_size'], use_aug=True)
model  = MedSAMMultiTask(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
    freeze_encoder=False,
)
crit   = make_criterion(cfg_msam)

r09 = train_model(model, tl, vl, crit, cfg_msam,
                  'exp09_medsam_fullaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP09 (MAIN MODEL): DSC={r09["best_dsc"]:.4f} | AUC={r09["best_auc"]:.4f}')
print(f'Full aug gain (MedSAM): '
      f'ΔDSC={r09["best_dsc"]-r07["best_dsc"]:+.4f} | '
      f'ΔAUC={r09["best_auc"]-r07["best_auc"]:+.4f}')

BEAT_DSC = r09['best_dsc'] >= 0.841
BEAT_AUC = r09['best_auc'] >= 0.866
print(f'\nBeats FedBCa DSC 0.841: {"✓ YES" if BEAT_DSC else "✗ no"}')
print(f'Beats FedBCa AUC 0.866: {"✓ YES" if BEAT_AUC else "✗ no"}')

  Train batches: 571 | Val batches: 67


c:\Users\Mayank\miniconda3\envs\medsam_fedbca\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


[MedSAM] resized pos_embed -> torch.Size([1, 16, 16, 768])
[MedSAM-MultiTask] in_ch=5 | freeze_enc=False | params=88,483,004

[exp09_medsam_fullaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 200
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/200 | Loss=1.6252 (seg=1.912 cls=0.000) | tDSC=0.1140 AUC=0.3304 comb=0.2114★ | LR=2.70e-06 | cls_w=0.000


...

Ep 160/200 | Loss=0.8458 (seg=0.964 cls=0.176) | tDSC=0.7158 AUC=0.6607 comb=0.6910 | LR=1.19e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 165/200 | Loss=0.8375 (seg=0.954 cls=0.179) | tDSC=0.7171 AUC=0.6786 comb=0.6997 | LR=1.00e-06 | cls_w=0.150

[EARLY STOP] No improvement for 80 epochs.


C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\utils\trainer.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_bes


[exp09_medsam_fullaug] Done.
  Best DSC : 0.7075
  Best AUC : 0.7054
  Combined : 0.7065

EXP09 (MAIN MODEL): DSC=0.7075 | AUC=0.7054
Full aug gain (MedSAM): ΔDSC=+0.0432 | ΔAUC=-0.1160

Beats FedBCa DSC 0.841: ✗ no
Beats FedBCa AUC 0.866: ✗ no
